# Feature Selection

## Objective

In this notebook, we will practice:

### Filter Methods
- Correlation Analysis
- Chi-Square Test
- ANOVA

### Wrapper Methods
- Recursive Feature Elimination (RFE)
- Forward Selection
- Backward Selection

### Embedded Methods
- Lasso
- Ridge
- Random Forest Feature Importance

> A randomly generated `practice_target` is used only to learn the Feature Selection workflow. Results must not be interpreted as real medical findings.

In [1]:
import numpy as np
import pandas as pd

from sklearn.feature_selection import (
    chi2,
    f_classif,
    RFE,
    SequentialFeatureSelector
)

from sklearn.linear_model import (
    LogisticRegression,
    Lasso,
    Ridge
)

from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler

In [2]:
df = pd.read_csv(
    "../heart_failure_clinical_records_dataset-selected-columns.csv"
)

df.head()

,age,anaemia,creatinine_phosphokinase,diabetes,ejection_fraction,high_blood_pressure,platelets,serum_creatinine,serum_sodium,sex
0,75.0,0,582,0,20,1,265000.00,1.9,130,1
1,55.0,0,7861,0,38,0,263358.03,1.1,136,1
2,65.0,0,146,0,20,0,162000.00,1.3,129,1
3,50.0,1,111,0,20,0,210000.00,1.9,137,1
4,65.0,1,160,1,20,0,327000.00,2.7,116,0


In [3]:
# Create reproducible random target
np.random.seed(42)

df_practice = df.copy()

df_practice["practice_target"] = np.random.randint(
    0,
    2,
    size=len(df_practice)
)

df_practice[
    ["practice_target"]
].value_counts()

practice_target
0                  151
1                  148
Name: count, dtype: int64

In [4]:
X = df_practice.drop(
    columns=["practice_target"]
)

y = df_practice["practice_target"]

print("Features Shape:", X.shape)
print("Target Shape:", y.shape)

Features Shape: (299, 10)
Target Shape: (299,)


# Filter Methods

Filter Methods use statistical techniques to evaluate features without repeatedly training different feature subsets.

In [5]:
correlation = (
    df_practice
    .corr(numeric_only=True)["practice_target"]
    .drop("practice_target")
    .sort_values(ascending=False)
)

correlation

serum_sodium                0.058370
platelets                   0.045699
anaemia                     0.042506
high_blood_pressure         0.042416
ejection_fraction           0.017339
sex                         0.013639
age                        -0.006431
creatinine_phosphokinase   -0.042755
diabetes                   -0.066086
serum_creatinine           -0.073541
Name: practice_target, dtype: float64

In [6]:
#chi square test
chi_scores, chi_p_values = chi2(
    X,
    y
)

chi_results = pd.DataFrame({
    "Feature": X.columns,
    "Chi2 Score": chi_scores,
    "P-Value": chi_p_values
}).sort_values(
    by="Chi2 Score",
    ascending=False
)

chi_results

,Feature,Chi2 Score,P-Value
6,platelets,22604.907672,0.000000e+00
2,creatinine_phosphokinase,881.423737,1.071825e-193
7,serum_creatinine,1.237425,2.659679e-01
3,diabetes,0.759924,3.833522e-01
5,high_blood_pressure,0.349035,5.546598e-01
4,ejection_fraction,0.329502,5.659523e-01
1,anaemia,0.307150,5.794342e-01
8,serum_sodium,0.144686,7.036666e-01
0,age,0.028663,8.655587e-01
9,sex,0.019532,8.888522e-01


In [7]:
# anova 
anova_scores, anova_p_values = f_classif(
    X,
    y
)

anova_results = pd.DataFrame({
    "Feature": X.columns,
    "F-Score": anova_scores,
    "P-Value": anova_p_values
}).sort_values(
    by="F-Score",
    ascending=False
)

anova_results

,Feature,F-Score,P-Value
7,serum_creatinine,1.614991,0.204786
3,diabetes,1.302802,0.254621
8,serum_sodium,1.015347,0.314446
6,platelets,0.621559,0.431098
2,creatinine_phosphokinase,0.543903,0.461402
1,anaemia,0.537581,0.464015
5,high_blood_pressure,0.535311,0.464960
4,ejection_fraction,0.089319,0.765254
9,sex,0.055258,0.814317
0,age,0.012283,0.911826


# Wrapper Methods

Wrapper Methods use a Machine Learning model to evaluate and select feature subsets.

In [8]:
model = LogisticRegression(
    max_iter=2000
)

In [9]:
rfe = RFE(
    estimator=model,
    n_features_to_select=5
)

rfe.fit(
    X,
    y
)

rfe_results = pd.DataFrame({
    "Feature": X.columns,
    "Selected": rfe.support_,
    "Ranking": rfe.ranking_
})

rfe_results.sort_values("Ranking")

,Feature,Selected,Ranking
1,anaemia,True,1
3,diabetes,True,1
7,serum_creatinine,True,1
5,high_blood_pressure,True,1
9,sex,True,1
8,serum_sodium,False,2
4,ejection_fraction,False,3
0,age,False,4
2,creatinine_phosphokinase,False,5
6,platelets,False,6


In [10]:
forward_selector = SequentialFeatureSelector(
    model,
    n_features_to_select=5,
    direction="forward",
    cv=5
)

forward_selector.fit(
    X,
    y
)

forward_features = X.columns[
    forward_selector.get_support()
]

print("Selected Features:")
print(forward_features.tolist())

Selected Features:
['anaemia', 'creatinine_phosphokinase', 'diabetes', 'platelets', 'serum_creatinine']


In [12]:
backward_selector = SequentialFeatureSelector(
    model,
    n_features_to_select=5,
    direction="backward",
    cv=5
)

backward_selector.fit(
    X,
    y
)

backward_features = X.columns[
    backward_selector.get_support()
]

print("Selected Features:")
print(backward_features.tolist())

f:\machine-learning-beginner-to-advanced\myenv\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 2000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=2000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
f:\machine-learning-beginner-to-advanced\myenv\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 2000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=2000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/

Selected Features:
['anaemia', 'creatinine_phosphokinase', 'diabetes', 'platelets', 'serum_creatinine']


# Embedded Methods

Embedded Methods determine feature influence during model training.

In [13]:
scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

X_scaled = pd.DataFrame(
    X_scaled,
    columns=X.columns
)

X_scaled.head()

,age,anaemia,creatinine_phosphokinase,diabetes,ejection_fraction,high_blood_pressure,platelets,serum_creatinine,serum_sodium,sex
0,1.192945,-0.871105,0.000166,-0.847579,-1.530560,1.359272,1.681648e-02,0.490057,-1.504036,0.735688
1,-0.491279,-0.871105,7.514640,-0.847579,-0.007077,-0.735688,7.535660e-09,-0.284552,-0.141976,0.735688
2,0.350833,-0.871105,-0.449939,-0.847579,-1.530560,-0.735688,-1.038073e+00,-0.090900,-1.731046,0.735688
3,-0.912335,1.147968,-0.486071,-0.847579,-1.530560,-0.735688,-5.464741e-01,0.490057,0.085034,0.735688
4,0.350833,1.147968,-0.435486,1.179830,-1.530560,-0.735688,6.517986e-01,1.264666,-4.682176,-1.359272


In [14]:
lasso = Lasso(
    alpha=0.01
)

lasso.fit(
    X_scaled,
    y
)

lasso_results = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": lasso.coef_
})

lasso_results[
    "Absolute Coefficient"
] = lasso_results[
    "Coefficient"
].abs()

lasso_results.sort_values(
    by="Absolute Coefficient",
    ascending=False
)

,Feature,Coefficient,Absolute Coefficient
7,serum_creatinine,-0.025903,0.025903
3,diabetes,-0.024379,0.024379
6,platelets,0.013608,0.013608
8,serum_sodium,0.011142,0.011142
2,creatinine_phosphokinase,-0.010504,0.010504
1,anaemia,0.010092,0.010092
5,high_blood_pressure,0.008549,0.008549
0,age,-0.000000,0.000000
4,ejection_fraction,0.000000,0.000000
9,sex,0.000000,0.000000


In [15]:
ridge = Ridge(
    alpha=1.0
)

ridge.fit(
    X_scaled,
    y
)

ridge_results = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": ridge.coef_
})

ridge_results[
    "Absolute Coefficient"
] = ridge_results[
    "Coefficient"
].abs()

ridge_results.sort_values(
    by="Absolute Coefficient",
    ascending=False
)

,Feature,Coefficient,Absolute Coefficient
7,serum_creatinine,-0.034470,0.034470
3,diabetes,-0.033680,0.033680
6,platelets,0.024763,0.024763
2,creatinine_phosphokinase,-0.019948,0.019948
1,anaemia,0.019847,0.019847
5,high_blood_pressure,0.018093,0.018093
8,serum_sodium,0.017235,0.017235
9,sex,0.011473,0.011473
0,age,-0.005038,0.005038
4,ejection_fraction,0.003343,0.003343


In [16]:
random_forest = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

random_forest.fit(
    X,
    y
)

rf_importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": random_forest.feature_importances_
}).sort_values(
    by="Importance",
    ascending=False
)

rf_importance

,Feature,Importance
2,creatinine_phosphokinase,0.178743
6,platelets,0.169748
0,age,0.165000
8,serum_sodium,0.134853
7,serum_creatinine,0.126885
4,ejection_fraction,0.108617
1,anaemia,0.031763
3,diabetes,0.029780
5,high_blood_pressure,0.029573
9,sex,0.025038


In [17]:
selected_summary = pd.DataFrame({
    "Feature": X.columns,
    "RFE Selected": rfe.support_,
    "Forward Selected": X.columns.isin(
        forward_features
    ),
    "Backward Selected": X.columns.isin(
        backward_features
    )
})

selected_summary

,Feature,RFE Selected,Forward Selected,Backward Selected
0,age,False,False,False
1,anaemia,True,True,True
2,creatinine_phosphokinase,False,True,True
3,diabetes,True,True,True
4,ejection_fraction,False,False,False
5,high_blood_pressure,True,False,False
6,platelets,False,True,True
7,serum_creatinine,True,True,True
8,serum_sodium,False,False,False
9,sex,True,False,False


In [18]:
print("Original Shape:", df.shape)

print(
    "practice_target in Original Dataset:",
    "practice_target" in df.columns
)

Original Shape: (299, 10)
practice_target in Original Dataset: False


# Summary

In this notebook, we practiced:

## Filter Methods
- Correlation Analysis
- Chi-Square Test
- ANOVA

## Wrapper Methods
- Recursive Feature Elimination (RFE)
- Forward Selection
- Backward Selection

## Embedded Methods
- Lasso
- Ridge
- Random Forest Feature Importance

## Important Note

The `practice_target` was randomly generated only for learning the Feature Selection workflow.

Therefore:

- Selected features are not medically meaningful.
- Feature rankings should not be interpreted as real predictors.
- A real Machine Learning project must use the actual target variable.

## Key Learning

- Filter Methods use statistical relationships.
- Wrapper Methods evaluate feature subsets using a model.
- Embedded Methods select or rank features during model training.
- Feature Selection should eventually be performed using real training data and a real target.